# NB_patient_360
Builds `adb_caresync.gold.patient_360` per the STTM *Silver to Gold* (sheet `patient_360`).
* **Grain:** one row per patient
* **Driving table:** `silver.patients` (all patients, no filters)
* **Joins:** LEFT JOIN `silver.hospitals`, `silver.insurance_providers`, and `silver.lab_results` aggregated by `patient_id`
* **Load type:** FULL rebuild (CREATE OR REPLACE TABLE ... AS SELECT)

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab test summary'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                AS total_tests,
    COUNT_IF(result_status = 'Abnormal')    AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')    AS critical_test_count,
    MAX(test_date)                          AS last_test_date
  FROM adb_caresync.silver.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                                        AS patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                                    AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  CAST(p.registered_hospital_id AS INT)                                            AS hospital_id,
  h.hospital_name,
  CAST(p.insurance_provider_id AS INT)                                             AS insurance_provider_id,
  ip.provider_name                                                                 AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)                                         AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT)                                 AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT)                                 AS critical_test_count,
  la.last_test_date
FROM adb_caresync.silver.patients p
LEFT JOIN adb_caresync.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id

In [0]:
updated_df=spark.read.table("adb_caresync.gold.patient_360")
record_count=updated_df.count()
dbutils.notebook.exit(str(record_count))
